# 🎯 Análisis Final: Aprendizaje No Supervisado y Conclusiones

**Evaluación 2 — Aprendizaje No Supervisado (30%)**

Este notebook cubre:
1. K-Means Clustering de jugadores FIDE
2. Método del Codo y selección del K óptimo
3. PCA para visualización 2D
4. Métricas de clustering (Silhouette, Calinski-Harabasz, Davies-Bouldin)
5. Interpretación de clusters y conclusiones generales del proyecto

In [ ]:
%load_ext kedro.ipython

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42

In [ ]:
df = catalog.load('fide_preprocessed_data')

CLUSTER_FEATURES = ['rating_std_avg', 'rating_change', 'total_months_active', 'age_approx']
available = [c for c in CLUSTER_FEATURES if c in df.columns]
X = df[available].dropna()

print(f'Features para clustering: {available}')
print(f'Registros: {X.shape}')

# Escalar
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 1. Método del Codo y Selección de K

In [ ]:
k_range = range(2, 11)
inertias = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f'K={k:2d} | Inertia: {km.inertia_:12.2f} | Silhouette: {sil:.4f}')

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'\nMejor K por Silhouette Score: {best_k}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Método del Codo
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(best_k, color='red', linestyle='--', label=f'K óptimo = {best_k}')
axes[0].set_title('Método del Codo')
axes[0].set_xlabel('Número de Clusters (K)')
axes[0].set_ylabel('Inercia')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Silhouette Score
axes[1].plot(list(k_range), silhouettes, 'rs-', linewidth=2, markersize=8)
axes[1].axvline(best_k, color='green', linestyle='--', label=f'K óptimo = {best_k}')
axes[1].set_title('Silhouette Score por K')
axes[1].set_xlabel('Número de Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Selección del Número Óptimo de Clusters', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 2. K-Means Final y Visualización con PCA

In [ ]:
# K-Means con el K óptimo
km_final = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
cluster_labels = km_final.fit_predict(X_scaled)

# PCA a 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f'Varianza explicada: PC1={pca.explained_variance_ratio_[0]:.4f}, '
      f'PC2={pca.explained_variance_ratio_[1]:.4f}, '
      f'Total={sum(pca.explained_variance_ratio_):.4f}')

In [ ]:
# Visualización 2D con PCA
fig, ax = plt.subplots(figsize=(12, 8))

scatter = ax.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=cluster_labels,
    cmap='viridis',
    alpha=0.5,
    s=10,
)

# Centroides
centroids_pca = pca.transform(km_final.cluster_centers_)
ax.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='red', marker='X', s=200, edgecolors='black', linewidths=2,
    label='Centroides',
)

ax.set_title(f'Clustering de Jugadores FIDE (K={best_k}) \u2014 Proyección PCA 2D')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.legend()
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

## 3. Métricas de Evaluación del Clustering

In [ ]:
sil = silhouette_score(X_scaled, cluster_labels)
cal = calinski_harabasz_score(X_scaled, cluster_labels)
dav = davies_bouldin_score(X_scaled, cluster_labels)

metrics_df = pd.DataFrame({
    'Métrica': ['Silhouette Score', 'Calinski-Harabasz Index', 'Davies-Bouldin Index'],
    'Valor': [sil, cal, dav],
    'Interpretación': [
        'Más alto = mejor (rango -1 a 1)',
        'Más alto = mejor separación entre clusters',
        'Más bajo = mejor (clusters más compactos)',
    ]
})

display(metrics_df.style.format({'Valor': '{:.4f}'}))

## 4. Interpretación de los Clusters

In [ ]:
# Estadísticas por cluster
X_with_clusters = X.copy()
X_with_clusters['cluster'] = cluster_labels

cluster_stats = X_with_clusters.groupby('cluster').agg(['mean', 'std', 'count'])
display(cluster_stats)

In [ ]:
# Distribución de clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
axes[0].pie(cluster_counts, labels=[f'Cluster {i}' for i in cluster_counts.index],
            autopct='%1.1f%%', colors=sns.color_palette('viridis', best_k))
axes[0].set_title('Distribución de Jugadores por Cluster')

# Boxplot de rating por cluster
if 'rating_std_avg' in X_with_clusters.columns:
    X_with_clusters.boxplot(column='rating_std_avg', by='cluster', ax=axes[1])
    axes[1].set_title('Rating Promedio por Cluster')
    axes[1].set_xlabel('Cluster')
    axes[1].set_ylabel('Rating Estándar Promedio')
    plt.suptitle('')

plt.tight_layout()
plt.show()

## 5. Conclusiones Generales del Proyecto

### Evaluación 1 — Calidad y Transformación de Datos
- Se cargaron exitosamente 4 datasets FIDE (~12M registros).
- Se limpiaron duplicados, nulos y outliers (IQR en rating_standard).
- Se integraron las tablas mediante joins y se crearon features derivadas.
- La validación post-transformación confirma la integridad del dataset final.

### Evaluación 2 — Machine Learning
- **Supervisado:** Se entrenaron 5 modelos de clasificación para predecir si un jugador es "experto" (ELO > 2000).
- **Evaluación:** Se aplicó validación cruzada 5-fold con múltiples métricas.
- **Optimización:** GridSearchCV mejoró los hiperparámetros de RF y GB.
- **No Supervisado:** K-Means reveló segmentos naturales de jugadores basados en rating, actividad y edad.

### Lecciones Aprendidas
- La modularidad de Kedro facilita la reproducibilidad y el mantenimiento del código.
- La combinación de técnicas supervisadas y no supervisadas enriquece el análisis.
- La semilla fija (`random_state=42`) garantiza resultados reproducibles.